# Cross-task behavior of the successful single-task checkpoint

This inference-only Modal experiment probes the successful step-4096
transformer checkpoint outside the one ARC task on which it was trained.
It does **not** expect the seven unrelated queries to be solved. Instead,
it asks how the checkpoint fails:

- Does it still produce a valid PAD-delimited grid?
- Does it infer a plausible output size?
- Does it imitate the learned 3x3 stripe transformation?
- Does any target structure survive on larger 10x10, 20x20, or 30x30 outputs?

All seven queries use the same initial latent noise. This isolates the
effect of NVARC conditioning while retaining the original 50-step linear
Euler trajectory. Only one sample per query is generated; this is a
qualitative out-of-distribution diagnosis, not a pass@k benchmark.

Prerequisites:

- a Modal GPU with at least 70 GiB VRAM;
- the `diffusion-arc-models` Volume mounted at `/mnt/diffusion-arc-models`;
- a Modal Secret containing `GITHUB_TOKEN`;
- `DIFFUSION_ARC_REVISION` set to the exact pushed commit containing this notebook.

In [ ]:
%uv pip uninstall torchvision

In [ ]:
from __future__ import annotations

import base64
from dataclasses import asdict, replace
from datetime import datetime, timezone
import hashlib
import importlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import re
import shutil
import subprocess
import sys
import textwrap
import time

TRAINING_RUN_ID = '20260827T011822050154Z'
TRAINING_STAGE = 'augmentation-generalization-overfit'
TRAINED_TASK_ID = '0d3d703e'
CHECKPOINT_STEP = 4096
GENERATION_SEED = 167
INFERENCE_STEPS = 50
MAX_PROMPT_TOKENS = 8192
MINIMUM_VRAM_GIB = 70.0
SOURCE_REVISION = "1a47688099912cb7acc477b471a0f83f95bf5415"
AUTO_SHUTDOWN = os.environ.get('DIFFUSION_ARC_AUTO_SHUTDOWN', '1') == '1'

PANEL_SPEC = (
    {
        'task_id': '3c9b0459', 'query_index': 0,
        'category': 'old-overfitting-pool', 'batch_group': 'short',
        'expected_shape': (3, 3),
        'reason': 'same output size as trained task, different rule',
    },
    {
        'task_id': '8be77c9e', 'query_index': 0,
        'category': 'old-overfitting-pool', 'batch_group': 'short',
        'expected_shape': (6, 3),
        'reason': 'vertical shape expansion',
    },
    {
        'task_id': '00576224', 'query_index': 0,
        'category': 'old-overfitting-pool', 'batch_group': 'short',
        'expected_shape': (6, 6),
        'reason': 'larger tiled pattern',
    },
    {
        'task_id': 'd631b094', 'query_index': 0,
        'category': 'old-overfitting-pool', 'batch_group': 'short',
        'expected_shape': (1, 5),
        'reason': 'extreme narrow output shape',
    },
    {
        'task_id': '60c09cac', 'query_index': 0,
        'category': 'size-probe', 'batch_group': 'short',
        'expected_shape': (10, 10),
        'reason': '10x10 output with short prompt and 800 PAD cells',
    },
    {
        'task_id': 'feca6190', 'query_index': 0,
        'category': 'size-probe', 'batch_group': 'medium',
        'expected_shape': (20, 20),
        'reason': '20x20 output with 500 PAD cells',
    },
    {
        'task_id': 'e4888269', 'query_index': 0,
        'category': 'size-probe', 'batch_group': 'long',
        'expected_shape': (30, 30),
        'reason': 'full 30x30 output without PAD cells',
    },
)
BATCH_GROUPS = ('short', 'medium', 'long')

REPOSITORY_URL = 'https://github.com/joshuakauling/diffusion-arc.git'
VOLUME_ROOT = Path('/mnt/diffusion-arc-models')
BASE_FLUX_CHECKPOINT = VOLUME_ROOT / 'flux2-klein-base-4b'
NVARC_CHECKPOINT = VOLUME_ROOT / 'nvarc-qwen3-4b-grids15-sft139'
ARC_DATA_ROOT = VOLUME_ROOT / 'data' / 'raw' / 'arc-agi-2'
ARC_DATA_MANIFEST = VOLUME_ROOT / 'data' / 'arc-agi-2-manifest.json'
TRAINING_RUN_ROOT = (
    VOLUME_ROOT / 'results' / 'single-task-augmentation-generalization-overfitting'
    / TRAINING_RUN_ID
)
TRAINING_REPORT_PATH = TRAINING_RUN_ROOT / 'report.json'
TRANSFORMER_CHECKPOINT = (
    TRAINING_RUN_ROOT / TRAINING_STAGE / f'flux-transformer-step-{CHECKPOINT_STEP:04d}'
)

EVALUATION_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
SOURCE_DIR = Path('/tmp') / f'diffusion-arc-cross-task-eval-{EVALUATION_ID}'
EVALUATION_ROOT = (
    VOLUME_ROOT / 'results' / 'single-task-checkpoint-cross-task-diagnosis'
    / EVALUATION_ID
)
EVALUATION_REPORT_PATH = EVALUATION_ROOT / 'report.json'
IMAGES_ROOT = EVALUATION_ROOT / 'images'

PINNED_PACKAGES = {
    'accelerate': '1.12.0',
    'diffusers': '0.39.0',
    'einops': '0.8.1',
    'Pillow': '11.3.0',
    'safetensors': '0.8.0',
    'torch': '2.13.0',
    'tqdm': '4.67.1',
    'transformers': '4.56.1',
}


def require_exact_commit(revision: str) -> None:
    if len(revision) != 40 or any(
        character not in '0123456789abcdefABCDEF' for character in revision
    ):
        raise ValueError(
            'DIFFUSION_ARC_REVISION must be an exact pushed 40-character commit'
        )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(8 * 1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def run_command(
    label: str,
    command: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
) -> str:
    completed = subprocess.run(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if completed.returncode != 0:
        tail = '\n'.join(completed.stdout.splitlines()[-50:])
        if tail:
            print(tail)
        raise RuntimeError(f'{label} failed with exit code {completed.returncode}')
    return completed.stdout.strip()


def installed_version(name: str) -> str | None:
    try:
        return importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:
        return None


require_exact_commit(SOURCE_REVISION)
required_roots = (
    VOLUME_ROOT,
    BASE_FLUX_CHECKPOINT,
    NVARC_CHECKPOINT,
    ARC_DATA_ROOT,
    ARC_DATA_MANIFEST,
    TRAINING_REPORT_PATH,
    TRANSFORMER_CHECKPOINT,
)
missing_roots = [str(path) for path in required_roots if not path.exists()]
if missing_roots:
    raise FileNotFoundError(f'Missing required Volume artifacts: {missing_roots}')
if EVALUATION_ROOT.exists():
    raise FileExistsError(EVALUATION_ROOT)
EVALUATION_ROOT.mkdir(parents=True)
IMAGES_ROOT.mkdir()

REPORT: dict[str, object] = {
    'schema_version': 1,
    'status': 'INITIALIZING',
    'evaluation_id': EVALUATION_ID,
    'question': (
        'How does the successful single-task step-4096 checkpoint fail on '
        'seven unrelated original ARC queries, including 10x10, 20x20, and '
        '30x30 outputs?'
    ),
    'interpretation_boundary': (
        'One shared-noise sample per query is a qualitative cross-task '
        'diagnosis, not a pass@k estimate and not evidence of task-level '
        'generalization.'
    ),
    'configuration': {
        'training_run_id': TRAINING_RUN_ID,
        'training_stage': TRAINING_STAGE,
        'trained_task_id': TRAINED_TASK_ID,
        'checkpoint_step': CHECKPOINT_STEP,
        'generation_seed': GENERATION_SEED,
        'shared_initial_noise': True,
        'inference_steps': INFERENCE_STEPS,
        'max_prompt_tokens': MAX_PROMPT_TOKENS,
        'requested_source_revision': SOURCE_REVISION,
        'automatic_shutdown': AUTO_SHUTDOWN,
        'panel_spec': [dict(item) for item in PANEL_SPEC],
    },
    'paths': {
        'training_report': str(TRAINING_REPORT_PATH),
        'transformer_checkpoint': str(TRANSFORMER_CHECKPOINT),
        'evaluation_report': str(EVALUATION_REPORT_PATH),
        'images': str(IMAGES_ROOT),
    },
    'failures': [],
}


def save_report() -> None:
    temporary = EVALUATION_REPORT_PATH.with_suffix('.json.tmp')
    temporary.write_text(
        json.dumps(REPORT, indent=2, sort_keys=True, allow_nan=False) + '\n',
        encoding='utf-8',
    )
    temporary.replace(EVALUATION_REPORT_PATH)
    if hasattr(os, 'sync'):
        os.sync()


def record_failure(stage: str, error: BaseException) -> None:
    REPORT['status'] = 'FAIL'
    REPORT['failed_stage'] = stage
    failures = REPORT.setdefault('failures', [])
    if isinstance(failures, list):
        failures.append({
            'stage': stage,
            'type': type(error).__name__,
            'message': str(error),
            'recorded_at_utc': datetime.now(timezone.utc).isoformat(),
        })
    save_report()


save_report()
print('Evaluation ID:', EVALUATION_ID)
print('Checkpoint:', TRANSFORMER_CHECKPOINT)
print('Requested source revision:', SOURCE_REVISION)
print('Report:', EVALUATION_REPORT_PATH)

## Fixed diagnostic panel

The four old-pool queries cover a same-size control and three different
output geometries. The size probes were selected deterministically for
exact 10x10, 20x20, and 30x30 targets with relatively compact prompts.

The `short` group is evaluated together. The longer 20x20 and 30x30
prompts are evaluated separately to avoid padding every NVARC context to
the longest sequence. The same initial latent tensor is repeated for all
seven queries across all three groups.

In [ ]:
try:
    SOURCE_DIR.mkdir(parents=True)
    run_command('git init', ['git', 'init'], cwd=SOURCE_DIR)
    run_command(
        'add origin',
        ['git', 'remote', 'add', 'origin', REPOSITORY_URL],
        cwd=SOURCE_DIR,
    )

    github_token = os.environ.get('GITHUB_TOKEN', '').strip()
    if not github_token:
        raise RuntimeError('Attach a Modal Secret containing GITHUB_TOKEN')
    credential = base64.b64encode(
        f'x-access-token:{github_token}'.encode('utf-8')
    ).decode('ascii')
    git_environment = os.environ.copy()
    git_environment.pop('GITHUB_TOKEN', None)
    git_environment.update({
        'GIT_TERMINAL_PROMPT': '0',
        'GIT_CONFIG_COUNT': '1',
        'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraheader',
        'GIT_CONFIG_VALUE_0': f'AUTHORIZATION: basic {credential}',
    })
    try:
        run_command(
            'fetch source',
            ['git', 'fetch', '--depth=1', 'origin', SOURCE_REVISION],
            cwd=SOURCE_DIR,
            env=git_environment,
        )
        run_command(
            'checkout source',
            ['git', 'checkout', '--detach', 'FETCH_HEAD'],
            cwd=SOURCE_DIR,
        )
    finally:
        git_environment.clear()
        del credential

    RESOLVED_SOURCE_COMMIT = run_command(
        'resolve source', ['git', 'rev-parse', 'HEAD'], cwd=SOURCE_DIR
    )
    if RESOLVED_SOURCE_COMMIT.lower() != SOURCE_REVISION.lower():
        raise RuntimeError('Fetched source commit differs from requested revision')

    requirements = [
        f'{name}=={version}'
        for name, version in PINNED_PACKAGES.items()
        if installed_version(name) != version
    ]
    if requirements:
        run_command(
            'install pinned dependencies',
            [
                sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
                '--no-cache-dir', *requirements,
            ],
        )
    run_command(
        'install Diffusion-ARC source',
        [
            sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input',
            '--no-deps', '--no-build-isolation', '-e', str(SOURCE_DIR),
        ],
    )

    source_path = str((SOURCE_DIR / 'src').resolve())
    if source_path not in sys.path:
        sys.path.insert(0, source_path)
    importlib.invalidate_caches()
    import diffusion_arc

    test_environment = os.environ.copy()
    test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
    test_environment['PYTHONPATH'] = source_path
    test_started = time.perf_counter()
    test_process = subprocess.run(
        [
            sys.executable, '-B', '-m', 'unittest',
            'tests.inference.test_conditioning',
            'tests.inference.test_generation',
            'tests.arc_data.test_decoding',
            'tests.arc_data.test_task_loading',
            'tests.arc_data.test_prompt',
            'tests.arc_data.test_tokenization',
        ],
        cwd=SOURCE_DIR,
        env=test_environment,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        check=False,
    )
    if test_process.returncode != 0:
        print('\n'.join(test_process.stdout.splitlines()[-50:]))
        raise RuntimeError(f'Focused unit tests returned {test_process.returncode}')

    REPORT['source'] = {
        'repository': REPOSITORY_URL,
        'requested_revision': SOURCE_REVISION,
        'resolved_commit': RESOLVED_SOURCE_COMMIT,
        'path': str(SOURCE_DIR),
    }
    REPORT['packages'] = {
        name: installed_version(name) for name in PINNED_PACKAGES
    }
    REPORT['focused_unit_tests'] = {
        'returncode': 0,
        'seconds': time.perf_counter() - test_started,
    }
    REPORT['status'] = 'SOURCE_READY'
    save_report()
    print('Resolved source:', RESOLVED_SOURCE_COMMIT)
    print(f'Focused tests: PASS in {REPORT["focused_unit_tests"]["seconds"]:.1f}s')
except Exception as error:
    record_failure('source-bootstrap', error)
    raise

In [ ]:
try:
    training_report = json.loads(TRAINING_REPORT_PATH.read_text(encoding='utf-8'))
    if training_report.get('status') != 'COMPLETE':
        raise RuntimeError('Training report is not COMPLETE')
    if training_report.get('durability', {}).get('remote_content_verified') is not True:
        raise RuntimeError('Training report lacks verified remote durability')
    training_protocol = training_report.get('protocol', {})
    if (
        training_protocol.get('task_id') != TRAINED_TASK_ID
        or training_protocol.get('final_optimizer_step') != CHECKPOINT_STEP
    ):
        raise RuntimeError('Training task or final optimizer step differs')

    training_stage = training_report.get('stages', {}).get(TRAINING_STAGE, {})
    checkpoint = training_stage.get('saved_checkpoint')
    if training_stage.get('status') != 'COMPLETE' or not isinstance(checkpoint, dict):
        raise RuntimeError('Completed training stage or saved checkpoint is missing')
    if (
        checkpoint.get('step') != CHECKPOINT_STEP
        or checkpoint.get('state_scope') != 'transformer_only'
        or checkpoint.get('remote_file_sizes_verified') is not True
    ):
        raise RuntimeError('Checkpoint contract differs')
    if Path(checkpoint.get('path', '')).resolve() != TRANSFORMER_CHECKPOINT.resolve():
        raise RuntimeError('Configured checkpoint path differs from training report')

    expected_manifest = checkpoint.get('manifest')
    if not isinstance(expected_manifest, dict) or not expected_manifest:
        raise RuntimeError('Checkpoint file manifest is missing')
    actual_checkpoint_files = {
        str(path.relative_to(TRANSFORMER_CHECKPOINT)): path
        for path in sorted(TRANSFORMER_CHECKPOINT.rglob('*'))
        if path.is_file()
    }
    if set(actual_checkpoint_files) != set(expected_manifest):
        raise RuntimeError('Checkpoint file set differs from training report')
    for relative, path in actual_checkpoint_files.items():
        if path.stat().st_size != expected_manifest[relative]['bytes']:
            raise RuntimeError(f'Checkpoint file size differs: {relative}')
    if sha256_file(TRANSFORMER_CHECKPOINT / 'config.json') != expected_manifest[
        'config.json'
    ]['sha256']:
        raise RuntimeError('Checkpoint config hash differs')

    dataset_manifest = json.loads(ARC_DATA_MANIFEST.read_text(encoding='utf-8'))
    panel_task_ids = tuple(item['task_id'] for item in PANEL_SPEC)
    if TRAINED_TASK_ID in panel_task_ids or len(set(panel_task_ids)) != len(panel_task_ids):
        raise RuntimeError('Panel must contain seven unique tasks excluding trained task')
    for task_id in panel_task_ids:
        relative = f'training/{task_id}.json'
        task_path = ARC_DATA_ROOT / relative
        expected_hash = dataset_manifest.get('files', {}).get(relative)
        if expected_hash is None or not task_path.is_file():
            raise RuntimeError(f'Dataset manifest lacks panel task: {relative}')
        if sha256_file(task_path) != expected_hash:
            raise RuntimeError(f'Panel task differs from dataset manifest: {relative}')

    old_pool_ids = set(json.loads(
        (SOURCE_DIR / 'data' / 'splits' / 'overfitting_run_1' / 'train.json')
        .read_text(encoding='utf-8')
    ))
    configured_old_pool_ids = {
        item['task_id'] for item in PANEL_SPEC
        if item['category'] == 'old-overfitting-pool'
    }
    if not configured_old_pool_ids.issubset(old_pool_ids):
        raise RuntimeError('Configured old-pool tasks are not in overfitting_run_1')

    REPORT['training_checkpoint'] = {
        'training_source_commit': training_report.get('source', {}).get(
            'resolved_commit'
        ),
        'path': str(TRANSFORMER_CHECKPOINT),
        'step': checkpoint['step'],
        'state_scope': checkpoint['state_scope'],
        'manifest_file_count': len(expected_manifest),
        'all_file_sizes_reverified': True,
        'config_sha256_reverified': True,
        'weight_sha256_reverified': False,
        'weight_sha256_note': (
            'The completed training report already records verified remote content; '
            'this cost-capped inference run rechecks the file set and byte sizes.'
        ),
        'training_report_sha256': sha256_file(TRAINING_REPORT_PATH),
    }
    REPORT['dataset'] = {
        'manifest': str(ARC_DATA_MANIFEST),
        'manifest_sha256': sha256_file(ARC_DATA_MANIFEST),
        'panel_task_ids': list(panel_task_ids),
        'old_overfitting_pool_task_ids': sorted(configured_old_pool_ids),
        'size_probe_task_ids': [
            item['task_id'] for item in PANEL_SPEC
            if item['category'] == 'size-probe'
        ],
    }
    REPORT['status'] = 'ARTIFACTS_VERIFIED'
    save_report()
    print('Checkpoint file sizes verified:', len(actual_checkpoint_files))
    print('Panel dataset hashes verified:', len(panel_task_ids))
except Exception as error:
    record_failure('artifact-validation', error)
    raise

## Load the frozen inference stack

NVARC, the VAE, and the successful transformer checkpoint are loaded from
the mounted Volume. No optimizer is created and every model is frozen.

In [ ]:
try:
    import torch

    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is unavailable')
    if not torch.cuda.is_bf16_supported():
        raise RuntimeError('Selected CUDA device does not support BF16')
    DEVICE = torch.device('cuda:0')
    torch.cuda.set_device(DEVICE)
    properties = torch.cuda.get_device_properties(DEVICE)
    gpu_vram_gib = properties.total_memory / 2**30
    if gpu_vram_gib < MINIMUM_VRAM_GIB:
        raise RuntimeError(
            f'Requires at least {MINIMUM_VRAM_GIB:.0f} GiB VRAM; received '
            f'{properties.name} with {gpu_vram_gib:.1f} GiB'
        )

    os.environ['HF_HUB_OFFLINE'] = '1'
    os.environ['TRANSFORMERS_OFFLINE'] = '1'
    os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
    os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'
    os.environ['TOKENIZERS_PARALLELISM'] = 'false'

    from diffusion_arc.training.single_gpu_config import load_single_gpu_training_config
    from diffusion_arc.training.single_gpu_setup import (
        load_single_gpu_models,
        prepare_single_gpu_device,
        seed_single_gpu_process,
    )

    source_config = load_single_gpu_training_config(
        SOURCE_DIR / 'configs' / 'training' / 'single_gpu_overfitting_pilot.json'
    )
    RUN_CONFIG = replace(
        source_config,
        models=replace(
            source_config.models,
            flux_checkpoint=BASE_FLUX_CHECKPOINT,
            nvarc_checkpoint=NVARC_CHECKPOINT,
        ),
    )
    DEVICE = prepare_single_gpu_device(RUN_CONFIG)
    seed_single_gpu_process(GENERATION_SEED)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(DEVICE)

    def cuda_memory(label: str) -> dict[str, object]:
        torch.cuda.synchronize(DEVICE)
        return {
            'label': label,
            'allocated_gib': torch.cuda.memory_allocated(DEVICE) / 2**30,
            'reserved_gib': torch.cuda.memory_reserved(DEVICE) / 2**30,
            'peak_allocated_gib': torch.cuda.max_memory_allocated(DEVICE) / 2**30,
            'peak_reserved_gib': torch.cuda.max_memory_reserved(DEVICE) / 2**30,
        }

    load_started = time.perf_counter()
    models = load_single_gpu_models(
        RUN_CONFIG,
        device=DEVICE,
        transformer_checkpoint=TRANSFORMER_CHECKPOINT,
    )
    models.transformer.requires_grad_(False)
    load_seconds = time.perf_counter() - load_started
    if any(parameter.requires_grad for parameter in models.nvarc_runner.parameters()):
        raise RuntimeError('NVARC unexpectedly trainable')
    if any(parameter.requires_grad for parameter in models.vae_decoder.parameters()):
        raise RuntimeError('VAE unexpectedly trainable')
    if any(parameter.requires_grad for parameter in models.transformer.parameters()):
        raise RuntimeError('Evaluation transformer unexpectedly trainable')

    REPORT['environment'] = {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'cuda_runtime': torch.version.cuda,
        'gpu_name': properties.name,
        'gpu_vram_gib': gpu_vram_gib,
        'compute_capability': list(torch.cuda.get_device_capability(DEVICE)),
        'bf16_supported': torch.cuda.is_bf16_supported(),
    }
    REPORT['models'] = {
        'load_seconds': load_seconds,
        'transformer_parameters': sum(
            parameter.numel() for parameter in models.transformer.parameters()
        ),
        'nvarc_frozen': True,
        'vae_frozen': True,
        'transformer_frozen_for_inference': True,
        'memory_after_load': cuda_memory('after-model-load'),
    }
    REPORT['status'] = 'MODELS_READY'
    save_report()
    print(f'GPU: {properties.name} | {gpu_vram_gib:.1f} GiB')
    print(f'Models loaded in {load_seconds:.1f}s')
except Exception as error:
    record_failure('model-loading', error)
    raise

## Build target-free NVARC prompts

Only the selected test input and the task demonstrations are exposed to
the model. Test targets stay separate until scoring. Exact target sizes
are asserted against the frozen panel specification.

In [ ]:
try:
    from diffusion_arc.arc_data import (
        load_evaluation_task,
        load_evaluation_targets,
        render_evaluation_prompts,
        tokenize_nvarc_evaluation_sample,
    )
    from diffusion_arc.model.contracts import NVARC_MAX_TOKENS

    if NVARC_MAX_TOKENS != MAX_PROMPT_TOKENS:
        raise RuntimeError(
            f'NVARC token contract differs: {NVARC_MAX_TOKENS} versus '
            f'{MAX_PROMPT_TOKENS}'
        )

    panel_rows = []
    for spec in PANEL_SPEC:
        task_path = ARC_DATA_ROOT / 'training' / f'{spec["task_id"]}.json'
        evaluation_task = load_evaluation_task(task_path)
        evaluation_targets = load_evaluation_targets(task_path)
        rendered_by_query = {
            sample.query_index: sample
            for sample in render_evaluation_prompts(evaluation_task)
        }
        target_by_query = {
            target.query_index: target.output_grid
            for target in evaluation_targets.targets
        }
        query_index = spec['query_index']
        if query_index not in rendered_by_query or query_index not in target_by_query:
            raise RuntimeError(
                f'Missing query {query_index} for task {spec["task_id"]}'
            )
        rendered = rendered_by_query[query_index]
        tokenized = tokenize_nvarc_evaluation_sample(
            rendered,
            models.tokenizer,
            max_tokens=MAX_PROMPT_TOKENS,
        )
        expected_grid = target_by_query[query_index]
        expected_shape = (len(expected_grid), len(expected_grid[0]))
        if expected_shape != tuple(spec['expected_shape']):
            raise RuntimeError(
                f'Expected shape differs for {spec["task_id"]}: '
                f'{expected_shape} versus {spec["expected_shape"]}'
            )
        panel_rows.append({
            'spec': spec,
            'tokenized': tokenized,
            'expected_grid': expected_grid,
            'prompt_tokens': len(tokenized.input_ids),
            'prompt_sha256': hashlib.sha256(
                rendered.prompt.encode('utf-8')
            ).hexdigest(),
        })

    if len(panel_rows) != 7:
        raise RuntimeError(f'Expected seven panel rows, received {len(panel_rows)}')
    REPORT['panel'] = [
        {
            **dict(row['spec']),
            'expected_shape': list(row['spec']['expected_shape']),
            'prompt_tokens': row['prompt_tokens'],
            'prompt_sha256': row['prompt_sha256'],
            'target_hidden_from_prompt': True,
        }
        for row in panel_rows
    ]
    REPORT['status'] = 'PANEL_READY'
    save_report()
    for row in REPORT['panel']:
        print(
            f"{row['task_id']} q{row['query_index']} | "
            f"{row['expected_shape'][0]}x{row['expected_shape'][1]} | "
            f"{row['prompt_tokens']} tokens | {row['batch_group']}"
        )
except Exception as error:
    record_failure('panel-construction', error)
    raise

## Generate once with shared initial noise

The five short prompts form one batch; the 20x20 and 30x30 probes form
one-sample batches. Each group receives an exact repetition of the same
initial latent tensor generated from seed 167. The resulting differences
therefore come from conditioning and numerical batching effects, not from
different random starts.

In [ ]:
try:
    from collections import Counter

    from PIL import Image
    from tqdm.auto import tqdm

    from diffusion_arc.arc_data.palette import PAD_CLASS
    from diffusion_arc.inference import (
        continue_arc_batch,
        diagnose_exact_grids,
        linear_timestep_schedule,
        prepare_evaluation_text_inputs,
    )
    from diffusion_arc.model.contracts import (
        DEFAULT_FLUX_SPATIAL_SHAPE,
        TARGET_LATENT_CHANNELS,
    )

    CANVAS_SIZE = DEFAULT_FLUX_SPATIAL_SHAPE.height
    if CANVAS_SIZE != 30 or DEFAULT_FLUX_SPATIAL_SHAPE.width != 30:
        raise RuntimeError('Cross-task diagnosis requires the production 30x30 canvas')


    def safe_mean(values: list[float]) -> float | None:
        return None if not values else sum(values) / len(values)


    def region_metrics(class_canvas, expected_grid) -> dict[str, object]:
        canvas = class_canvas.detach().to(device='cpu', dtype=torch.int64)
        height = len(expected_grid)
        width = len(expected_grid[0])
        expected = torch.full_like(canvas, PAD_CLASS)
        expected[:height, :width] = torch.tensor(expected_grid, dtype=torch.int64)

        grid_mask = torch.zeros_like(canvas, dtype=torch.bool)
        grid_mask[:height, :width] = True
        padding_mask = ~grid_mask
        boundary_mask = torch.zeros_like(grid_mask)
        if height < CANVAS_SIZE:
            boundary_mask[height, : min(width + 1, CANVAS_SIZE)] = True
        if width < CANVAS_SIZE:
            boundary_mask[:height, width] = True
        outer_padding_mask = padding_mask & ~boundary_mask

        def accuracy(mask) -> float | None:
            count = int(mask.sum().item())
            if count == 0:
                return None
            return float((canvas[mask] == expected[mask]).float().mean().item())

        content_values = canvas[canvas != PAD_CLASS].tolist()
        return {
            'target_region_cell_accuracy': accuracy(grid_mask),
            'padding_cell_accuracy': accuracy(padding_mask),
            'boundary_padding_accuracy': accuracy(boundary_mask),
            'outer_padding_accuracy': accuracy(outer_padding_mask),
            'pad_inside_expected_grid': int(
                (canvas[grid_mask] == PAD_CLASS).sum().item()
            ),
            'non_pad_outside_expected_grid': int(
                (canvas[padding_mask] != PAD_CLASS).sum().item()
            ),
            'expected_grid_cells': int(grid_mask.sum().item()),
            'expected_padding_cells': int(padding_mask.sum().item()),
            'boundary_padding_cells': int(boundary_mask.sum().item()),
            'outer_padding_cells': int(outer_padding_mask.sum().item()),
            'exact_class_canvas': bool((canvas == expected).all().item()),
            'classified_non_pad_histogram': {
                str(key): value for key, value in sorted(Counter(content_values).items())
            },
        }


    def stripe_signature(prediction) -> dict[str, object]:
        if not prediction.valid or prediction.grid is None:
            return {
                'stripe_like': False,
                'stripe_orientation': None,
                'learned_3x3_stripe_mode': False,
            }
        grid = torch.tensor(prediction.grid, dtype=torch.int64)
        height, width = grid.shape
        distinct_colors = int(torch.unique(grid).numel())
        vertical = bool(
            height >= 2 and width >= 2
            and distinct_colors >= 2
            and (grid == grid[:1, :]).all().item()
        )
        horizontal = bool(
            height >= 2 and width >= 2
            and distinct_colors >= 2
            and (grid == grid[:, :1]).all().item()
        )
        orientation = (
            'both' if vertical and horizontal
            else 'vertical' if vertical
            else 'horizontal' if horizontal
            else None
        )
        return {
            'stripe_like': orientation is not None,
            'stripe_orientation': orientation,
            'learned_3x3_stripe_mode': bool(
                tuple(grid.shape) == (3, 3) and orientation is not None
            ),
        }


    schedule = linear_timestep_schedule(INFERENCE_STEPS, device=DEVICE)
    shared_generator = torch.Generator(device=DEVICE).manual_seed(GENERATION_SEED)
    shared_initial_noise = torch.randn(
        (
            1,
            DEFAULT_FLUX_SPATIAL_SHAPE.token_count,
            TARGET_LATENT_CHANNELS,
        ),
        device=DEVICE,
        dtype=torch.float32,
        generator=shared_generator,
    )
    shared_noise_sha256 = hashlib.sha256(
        shared_initial_noise.detach().to(device='cpu').numpy().tobytes()
    ).hexdigest()

    generation_records = []
    group_runtimes = []
    generation_started = time.perf_counter()
    for group_name in tqdm(BATCH_GROUPS, desc='Cross-task groups', unit='group'):
        group_rows = [
            row for row in panel_rows if row['spec']['batch_group'] == group_name
        ]
        if not group_rows:
            raise RuntimeError(f'Empty configured batch group: {group_name}')
        group_started = time.perf_counter()
        text_inputs = prepare_evaluation_text_inputs(
            [row['tokenized'] for row in group_rows],
            nvarc_runner=models.nvarc_runner,
        )
        initial_latents = shared_initial_noise.repeat(len(group_rows), 1, 1)
        result = continue_arc_batch(
            models.transformer,
            models.vae_decoder,
            text_inputs,
            initial_latents=initial_latents,
            timestep_schedule=schedule,
            compute_dtype=models.flux_compute_dtype,
        )
        diagnostics = diagnose_exact_grids(
            result.predictions,
            [row['expected_grid'] for row in group_rows],
        )
        group_seconds = time.perf_counter() - group_started
        group_runtimes.append({
            'group': group_name,
            'count': len(group_rows),
            'max_prompt_tokens': max(row['prompt_tokens'] for row in group_rows),
            'seconds': group_seconds,
        })

        for index, (row, prediction, diagnostic) in enumerate(
            zip(group_rows, result.predictions, diagnostics, strict=True)
        ):
            spec = row['spec']
            task_id = spec['task_id']
            filename = f'{task_id}-q{spec["query_index"]}-seed-{GENERATION_SEED}.png'
            rgb_image = result.rgb_images[index].permute(1, 2, 0).contiguous().numpy()
            Image.fromarray(rgb_image).save(IMAGES_ROOT / filename)
            padding = region_metrics(
                result.class_canvases[index], row['expected_grid']
            )
            stripe = stripe_signature(prediction)
            generation_records.append({
                'task_id': task_id,
                'query_index': spec['query_index'],
                'category': spec['category'],
                'batch_group': group_name,
                'reason': spec['reason'],
                'prompt_tokens': row['prompt_tokens'],
                'expected_grid': row['expected_grid'],
                'prediction_grid': prediction.grid,
                'class_canvas': result.class_canvases[index].tolist(),
                'image': str((IMAGES_ROOT / filename).relative_to(EVALUATION_ROOT)),
                'exact_grid_diagnostic': asdict(diagnostic),
                'padding_diagnostic': padding,
                'stripe_diagnostic': stripe,
            })

        del result, diagnostics, text_inputs, initial_latents
        torch.cuda.empty_cache()

    generation_seconds = time.perf_counter() - generation_started
    if len(generation_records) != len(PANEL_SPEC):
        raise RuntimeError('Generation record count differs from panel')

    valid_count = sum(
        record['exact_grid_diagnostic']['prediction_valid']
        for record in generation_records
    )
    shape_count = sum(
        record['exact_grid_diagnostic']['shape_match']
        for record in generation_records
    )
    exact_count = sum(
        record['exact_grid_diagnostic']['exact_grid']
        for record in generation_records
    )
    learned_mode_count = sum(
        record['stripe_diagnostic']['learned_3x3_stripe_mode']
        for record in generation_records
    )
    target_region_accuracies = [
        record['padding_diagnostic']['target_region_cell_accuracy']
        for record in generation_records
    ]
    boundary_accuracies = [
        record['padding_diagnostic']['boundary_padding_accuracy']
        for record in generation_records
        if record['padding_diagnostic']['boundary_padding_accuracy'] is not None
    ]
    outer_accuracies = [
        record['padding_diagnostic']['outer_padding_accuracy']
        for record in generation_records
        if record['padding_diagnostic']['outer_padding_accuracy'] is not None
    ]

    REPORT['generation'] = {
        'seconds': generation_seconds,
        'seed': GENERATION_SEED,
        'inference_steps': INFERENCE_STEPS,
        'shared_initial_noise': True,
        'shared_initial_noise_sha256': shared_noise_sha256,
        'group_runtimes': group_runtimes,
        'records': generation_records,
        'summary': {
            'count': len(generation_records),
            'valid_grid_count': valid_count,
            'correct_shape_count': shape_count,
            'exact_grid_count': exact_count,
            'learned_3x3_stripe_mode_count': learned_mode_count,
            'mean_target_region_cell_accuracy': safe_mean(
                target_region_accuracies
            ),
            'mean_boundary_padding_accuracy': safe_mean(boundary_accuracies),
            'mean_outer_padding_accuracy': safe_mean(outer_accuracies),
        },
        'memory': cuda_memory('generation-complete'),
    }
    REPORT['status'] = 'GENERATED'
    save_report()
    print(json.dumps(REPORT['generation']['summary'], indent=2))
    print(f'Generation completed in {generation_seconds:.1f}s')
except Exception as error:
    record_failure('generation', error)
    raise

## Compact GT / prediction / padding sheet

The red rectangle on the full 30x30 classified canvas marks the expected
target extent. White cells are classified as PAD. This makes stray
non-PAD cells outside the expected target immediately visible even when
strict boundary decoding reports an invalid grid.

In [ ]:
try:
    from PIL import ImageDraw, ImageFont
    from diffusion_arc.arc_data.palette import TARGET_PALETTE_RGB

    palette = [tuple(int(channel) for channel in row) for row in TARGET_PALETTE_RGB]
    font = ImageFont.load_default()
    label_width = 330
    panel_width = 280
    row_height = 310
    header_height = 44
    sheet_width = label_width + 3 * panel_width
    sheet_height = header_height + len(generation_records) * row_height
    sheet = Image.new('RGB', (sheet_width, sheet_height), 'white')
    draw = ImageDraw.Draw(sheet)


    def render_classes(values, *, box_size: int = 246) -> Image.Image:
        rows = [list(row) for row in values]
        height = len(rows)
        width = len(rows[0])
        cell = max(3, min(34, box_size // max(height, width)))
        image = Image.new('RGB', (width * cell, height * cell), (238, 238, 238))
        image_draw = ImageDraw.Draw(image)
        for row_index, row in enumerate(rows):
            for column_index, value in enumerate(row):
                x0 = column_index * cell
                y0 = row_index * cell
                image_draw.rectangle(
                    (x0, y0, x0 + cell - 1, y0 + cell - 1),
                    fill=palette[int(value)],
                    outline=(210, 210, 210),
                )
        return image


    def paste_centered(image: Image.Image, x: int, y: int) -> None:
        offset_x = x + (panel_width - image.width) // 2
        offset_y = y + 34 + (row_height - 42 - image.height) // 2
        sheet.paste(image, (offset_x, offset_y))


    headers = ('Task / diagnostics', 'Ground truth', 'Decoded extent', 'Full 30x30 canvas')
    header_x = (8, label_width + 8, label_width + panel_width + 8,
                label_width + 2 * panel_width + 8)
    for text_value, x in zip(headers, header_x, strict=True):
        draw.text((x, 14), text_value, fill='black', font=font)

    for row_index, record in enumerate(generation_records):
        y = header_height + row_index * row_height
        draw.line((0, y, sheet_width, y), fill=(190, 190, 190), width=1)
        exact = record['exact_grid_diagnostic']
        padding = record['padding_diagnostic']
        stripe = record['stripe_diagnostic']
        predicted_shape = exact['predicted_shape']
        label_lines = [
            f"{record['task_id']} q{record['query_index']} [{record['category']}]",
            f"expected={tuple(exact['expected_shape'])} predicted={predicted_shape}",
            f"valid={exact['prediction_valid']} shape={exact['shape_match']} exact={exact['exact_grid']}",
            f"target cells={padding['target_region_cell_accuracy']:.3f}",
            f"PAD boundary={padding['boundary_padding_accuracy']}",
            f"PAD outer={padding['outer_padding_accuracy']}",
            f"non-PAD outside={padding['non_pad_outside_expected_grid']}",
            f"stripe={stripe['stripe_orientation']} learned-mode={stripe['learned_3x3_stripe_mode']}",
        ]
        for line_number, label in enumerate(label_lines):
            draw.text((8, y + 12 + 24 * line_number), label, fill='black', font=font)

        expected_image = render_classes(record['expected_grid'])
        paste_centered(expected_image, label_width, y)

        full_canvas = record['class_canvas']
        if predicted_shape is None:
            predicted_values = full_canvas
        else:
            predicted_height = max(1, min(CANVAS_SIZE, predicted_shape[0]))
            predicted_width = max(1, min(CANVAS_SIZE, predicted_shape[1]))
            predicted_values = [
                row[:predicted_width] for row in full_canvas[:predicted_height]
            ]
        prediction_image = render_classes(predicted_values)
        paste_centered(prediction_image, label_width + panel_width, y)

        canvas_image = render_classes(full_canvas)
        expected_height, expected_width = exact['expected_shape']
        canvas_cell = canvas_image.width // CANVAS_SIZE
        canvas_draw = ImageDraw.Draw(canvas_image)
        canvas_draw.rectangle(
            (
                0,
                0,
                expected_width * canvas_cell - 1,
                expected_height * canvas_cell - 1,
            ),
            outline=(255, 0, 0),
            width=max(1, canvas_cell // 3),
        )
        paste_centered(canvas_image, label_width + 2 * panel_width, y)

    sheet_path = IMAGES_ROOT / 'cross-task-comparison-sheet.png'
    sheet.save(sheet_path)
    REPORT['visualization'] = {
        'comparison_sheet': str(sheet_path.relative_to(EVALUATION_ROOT)),
        'rows': len(generation_records),
        'columns': list(headers),
        'expected_extent_outline': 'red',
        'pad_color': 'white',
    }
    REPORT['status'] = 'COMPLETE'
    REPORT['completed_at_utc'] = datetime.now(timezone.utc).isoformat()
    save_report()

    summary_rows = []
    for record in generation_records:
        exact = record['exact_grid_diagnostic']
        padding = record['padding_diagnostic']
        stripe = record['stripe_diagnostic']
        summary_rows.append({
            'task': record['task_id'],
            'expected': exact['expected_shape'],
            'predicted': exact['predicted_shape'],
            'valid': exact['prediction_valid'],
            'shape': exact['shape_match'],
            'target_cell_acc': round(
                padding['target_region_cell_accuracy'], 4
            ),
            'boundary_pad_acc': (
                None if padding['boundary_padding_accuracy'] is None
                else round(padding['boundary_padding_accuracy'], 4)
            ),
            'outer_pad_acc': (
                None if padding['outer_padding_accuracy'] is None
                else round(padding['outer_padding_accuracy'], 4)
            ),
            'non_pad_outside': padding['non_pad_outside_expected_grid'],
            'stripe': stripe['stripe_orientation'],
            'learned_mode': stripe['learned_3x3_stripe_mode'],
        })
    print(json.dumps(summary_rows, indent=2))
    print('COMPLETE report:', EVALUATION_REPORT_PATH)
    print('Comparison sheet:', sheet_path)

    try:
        from IPython.display import Image as DisplayImage, display
        display(DisplayImage(filename=str(sheet_path), width=1100))
    except Exception as preview_error:
        print(f'Could not display comparison sheet: {preview_error}')
except Exception as error:
    record_failure('visualization-and-finalization', error)
    raise

## Interpretation boundary

Useful outcomes include invalid canvases, a collapse to the learned 3x3
stripe mode, correct padding with wrong content, or partial structural
transfer. None of these seven single samples estimates task-level ARC
accuracy. A second seed or a base-checkpoint comparison is intentionally
excluded to keep this final diagnosis inexpensive.

In [ ]:
if not AUTO_SHUTDOWN:
    print('Automatic shutdown disabled; stop the kernel manually when finished.')
elif REPORT.get('status') != 'COMPLETE':
    raise RuntimeError(
        f"Refusing shutdown because report status is {REPORT.get('status')!r}"
    )
else:
    modal_task_id = os.environ.get('MODAL_TASK_ID', '').strip()
    modal_cli = shutil.which('modal')
    if not re.fullmatch(r'ta-[A-Za-z0-9-]+', modal_task_id):
        print('WARNING: MODAL_TASK_ID is unavailable; stop the container manually.')
    elif modal_cli is None:
        print('WARNING: Modal CLI is unavailable; stop the container manually.')
    else:
        shutdown_code = (
            'import subprocess, sys, time; time.sleep(15); '
            "subprocess.run(['modal', 'container', 'stop', '--yes', "
            'sys.argv[1]], check=False)'
        )
        subprocess.Popen(
            [sys.executable, '-c', shutdown_code, modal_task_id],
            stdin=subprocess.DEVNULL,
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            start_new_session=True,
            close_fds=True,
        )
        print('Evaluation vollständig gespeichert.')
        print(f'Modal-Container {modal_task_id} wird in 15 Sekunden beendet.')